In [1]:

# ================================================================
# ADAPTIVE KNOWLEDGE INTELLIGENCE SYSTEM
# Complete single-cell implementation with deep visualizations
# ================================================================

import sys, subprocess, importlib.util, warnings, re
from datetime import datetime
from itertools import combinations

warnings.filterwarnings("ignore")

# 1. Install required packages
packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "plotly": "plotly",
    "networkx": "networkx",
    "sklearn": "scikit-learn"
}
missing = [
    package for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])

import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

# 2. Configuration
SEED = 42
TODAY = pd.Timestamp.today().normalize()
HALF_LIFE = 180
REVIEW_THRESHOLD = 60
np.random.seed(SEED)

# 3. Create demonstration knowledge dataset
# Synthetic examples only; these are not verified factual records.
examples = [
    ("Software", "Path objects support filesystem operations.",
     "Official documentation", 5, 12),
    ("Software", "Path objects support filesystem operations.",
     "Internal notes", 3, 230),
    ("Security", "The sample service accepts requests without authentication.",
     "Legacy specification", 2, 500),
    ("Security", "The sample service requires authentication for every request.",
     "Security review", 4, 20),
    ("Biology", "A gene observed in two species proves direct transfer.",
     "Unreviewed note", 1, 650),
    ("Biology", "Shared gene occurrence alone cannot establish a direct transfer route.",
     "Methods note", 4, 45),
    ("Configuration", "The review threshold is fixed in configuration.",
     "Configuration guide", 4, 5),
    ("Configuration", "The review threshold is automatically learned from feedback.",
     "Roadmap note", 2, 310),
    ("Data Quality", "Every record should have a traceable source identifier.",
     "Data standards", 5, 30),
    ("Statistics", "Correlation proves that one variable causes another.",
     "Unreviewed note", 1, 420),
    ("Statistics", "Correlation alone does not establish causation.",
     "Statistics reference", 5, 22),
    ("Machine Learning", "A high training score guarantees good unseen performance.",
     "Old experiment note", 2, 390),
    ("Machine Learning", "Held-out evaluation estimates generalization performance.",
     "Evaluation protocol", 5, 18),
    ("API", "The API returns a JSON response for valid requests.",
     "API specification", 4, 12),
    ("API", "The API returns an XML response for valid requests.",
     "Old API note", 2, 260),
]

records = []
for i, (topic, claim, source, quality, age) in enumerate(examples, 1):
    records.append({
        "claim_id": f"C{i:03d}",
        "topic": topic,
        "claim": claim,
        "source": source,
        "source_quality": quality,
        "updated_at": TODAY - pd.Timedelta(days=age),
        "importance": np.random.randint(1, 6),
        "evidence": f"Synthetic evidence note for example {i}."
    })

extra_topics = [
    "Software", "Security", "Biology", "Configuration",
    "Data Quality", "Statistics", "Machine Learning", "API"
]

for i in range(16, 101):
    topic = extra_topics[(i * 7) % len(extra_topics)]
    age = int(np.random.choice([
        np.random.randint(0, 100),
        np.random.randint(100, 260),
        np.random.randint(260, 900)
    ]))
    quality = int(np.random.choice(
        [1, 2, 3, 4, 5],
        p=[0.08, 0.15, 0.25, 0.32, 0.20]
    ))
    records.append({
        "claim_id": f"C{i:03d}",
        "topic": topic,
        "claim": f"Illustrative knowledge statement {i} about {topic.lower()}.",
        "source": np.random.choice([
            "Official documentation", "Research paper",
            "Internal note", "Technical report", "Dataset metadata"
        ]),
        "source_quality": quality,
        "updated_at": TODAY - pd.Timedelta(days=age),
        "importance": np.random.randint(1, 6),
        "evidence": f"Synthetic evidence note for example {i}."
    })

df = pd.DataFrame(records)

# 4. Data validation
df["updated_at"] = pd.to_datetime(df["updated_at"], errors="coerce")
df["source_quality"] = pd.to_numeric(
    df["source_quality"], errors="coerce"
).fillna(3).clip(1, 5)

df["importance"] = pd.to_numeric(
    df["importance"], errors="coerce"
).fillna(3).clip(1, 5)

df["age_days"] = (
    TODAY - df["updated_at"]
).dt.days.clip(lower=0).fillna(9999)

assert df["claim_id"].is_unique
assert df["claim"].notna().all()

# 5. Knowledge freshness and health scoring
df["freshness_score"] = (
    100 * np.power(0.5, df["age_days"] / HALF_LIFE)
).clip(0, 100)

df["source_score"] = df["source_quality"] * 20

df["health_score"] = (
    0.65 * df["freshness_score"] +
    0.35 * df["source_score"]
).clip(0, 100)

df["staleness_risk"] = 100 - df["freshness_score"]

# 6. Candidate contradiction screening
# This is a transparent rule-based baseline, not semantic proof.
opposing_phrases = [
    ("without authentication", "requires authentication"),
    ("requires authentication", "without authentication"),
    ("proves direct transfer", "cannot establish a direct transfer"),
    ("proves direct transfer", "shared gene occurrence alone"),
    ("correlation proves", "correlation alone does not"),
    ("training score guarantees", "held-out evaluation"),
    ("json response", "xml response"),
]

conflict_pairs = []

for a, b in combinations(df.to_dict("records"), 2):
    if a["topic"] != b["topic"]:
        continue

    text_a = a["claim"].lower()
    text_b = b["claim"].lower()

    flagged = any(
        phrase_a in text_a and phrase_b in text_b
        for phrase_a, phrase_b in opposing_phrases
    )

    if flagged:
        conflict_pairs.append({
            "claim_a": a["claim_id"],
            "claim_b": b["claim_id"],
            "topic": a["topic"],
            "claim_a_text": a["claim"],
            "claim_b_text": b["claim"],
            "review_note": "Compare context, dates and source evidence."
        })

conflicts = pd.DataFrame(conflict_pairs)

conflict_ids = set()
if not conflicts.empty:
    conflict_ids = set(conflicts["claim_a"]) | set(conflicts["claim_b"])

df["potential_conflict"] = df["claim_id"].isin(conflict_ids)

# 7. Explainable review recommendations
reasons = []

for _, row in df.iterrows():
    items = []

    if row["age_days"] > HALF_LIFE:
        items.append("Old verification date")

    if row["source_quality"] <= 2:
        items.append("Low source-quality rating")

    if row["potential_conflict"]:
        items.append("Potential contradiction")

    if row["health_score"] < REVIEW_THRESHOLD:
        items.append("Low health score")

    reasons.append("; ".join(items))

df["review_reason"] = reasons
df["needs_review"] = df["review_reason"].ne("")

df["priority_score"] = (
    0.55 * (100 - df["health_score"]) +
    0.25 * df["importance"] * 20 +
    0.20 * df["potential_conflict"].astype(int) * 100
).clip(0, 100)

df["priority_band"] = pd.cut(
    df["priority_score"],
    bins=[-1, 34, 64, 100],
    labels=["Low", "Medium", "High"]
).astype(str)

# 8. Aggregate topic-level analytics
topic_summary = df.groupby("topic", as_index=False).agg(
    claim_count=("claim_id", "count"),
    mean_health=("health_score", "mean"),
    mean_freshness=("freshness_score", "mean"),
    mean_age_days=("age_days", "mean"),
    review_count=("needs_review", "sum"),
    conflict_count=("potential_conflict", "sum"),
    mean_source_quality=("source_quality", "mean")
)

topic_summary["review_rate_pct"] = (
    100 * topic_summary["review_count"] /
    topic_summary["claim_count"]
)

# 9. Anomaly detection on topic-level metrics
from sklearn.ensemble import IsolationForest

features = [
    "mean_health", "mean_freshness",
    "mean_age_days", "review_rate_pct",
    "mean_source_quality"
]

X = topic_summary[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

detector = IsolationForest(
    n_estimators=150,
    contamination=0.20,
    random_state=SEED
)

topic_summary["anomaly_label"] = detector.fit_predict(X)
topic_summary["anomaly_score"] = -detector.score_samples(X)
topic_summary["anomaly_flag"] = (
    topic_summary["anomaly_label"] == -1
)

# 10. Build an evidence relationship graph
G = nx.Graph()

for _, row in df.iterrows():
    claim_node = f"Claim:{row['claim_id']}"
    topic_node = f"Topic:{row['topic']}"
    source_node = f"Source:{row['source']}"

    G.add_node(
        claim_node, kind="Claim",
        label=row["claim_id"],
        health=float(row["health_score"])
    )
    G.add_node(topic_node, kind="Topic", label=row["topic"])
    G.add_node(source_node, kind="Source", label=row["source"])

    G.add_edge(claim_node, topic_node, relation="belongs to topic")
    G.add_edge(claim_node, source_node, relation="has source")

for _, row in conflicts.iterrows():
    G.add_edge(
        f"Claim:{row['claim_a']}",
        f"Claim:{row['claim_b']}",
        relation="candidate conflict"
    )

# 11. Dashboard heading
display(Markdown("""
# Adaptive Knowledge Intelligence
### Evidence monitoring, freshness analysis and review prioritization

**Important:** This notebook uses synthetic demonstration data.
The scores are heuristics, not probabilities that claims are true.
"""))

# 12. KPI cards
kpis = [
    ("Claims tracked", len(df)),
    ("Need review", int(df["needs_review"].sum())),
    ("Candidate conflict pairs", len(conflicts)),
    ("Mean health score", round(df["health_score"].mean(), 1))
]

fig = go.Figure()

for i, (label, value) in enumerate(kpis):
    fig.add_trace(go.Indicator(
        mode="number",
        value=value,
        title={"text": label},
        domain={"row": 0, "column": i},
        number={"font": {"size": 30}}
    ))

fig.update_layout(
    grid={"rows": 1, "columns": 4, "pattern": "independent"},
    height=150,
    margin=dict(l=10, r=10, t=30, b=10)
)
fig.show()

# 13. Health score distribution
fig = px.histogram(
    df,
    x="health_score",
    nbins=20,
    color="needs_review",
    title="1. Knowledge Health Score Distribution",
    hover_data=["claim_id", "topic", "age_days", "source_quality"]
)
fig.update_layout(bargap=0.12)
fig.show()

# 14. Evidence age versus source quality
fig = px.scatter(
    df,
    x="age_days",
    y="source_quality",
    size="importance",
    color="health_score",
    hover_name="claim_id",
    hover_data=["topic", "claim", "source", "review_reason"],
    title="2. Evidence Freshness vs Source Quality",
    labels={
        "age_days": "Days since verification",
        "source_quality": "Source quality (1–5)",
        "health_score": "Health score"
    }
)
fig.show()

# 15. Review rate by topic
fig = px.bar(
    topic_summary.sort_values(
        "review_rate_pct", ascending=False
    ),
    x="topic",
    y="review_rate_pct",
    color="mean_health",
    hover_data=[
        "claim_count", "review_count",
        "conflict_count", "mean_age_days"
    ],
    title="3. Review Rate by Topic",
    labels={
        "review_rate_pct": "Claims requiring review (%)",
        "mean_health": "Mean health"
    }
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

# 16. Freshness decay curve
days = np.arange(0, 1001)
freshness = 100 * (0.5 ** (days / HALF_LIFE))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=days, y=freshness,
    mode="lines",
    name="Freshness score",
    line=dict(width=3)
))
fig.add_hline(
    y=REVIEW_THRESHOLD,
    line_dash="dash",
    annotation_text="Illustrative review threshold"
)
fig.update_layout(
    title="4. Knowledge Freshness Decay",
    xaxis_title="Days since verification",
    yaxis_title="Freshness score",
    yaxis_range=[0, 105]
)
fig.show()

# 17. Topic-level anomaly analysis
fig = px.scatter(
    topic_summary,
    x="mean_age_days",
    y="review_rate_pct",
    size="claim_count",
    color="anomaly_flag",
    text="topic",
    hover_data=[
        "mean_health", "mean_freshness",
        "anomaly_score", "conflict_count"
    ],
    title="5. Topic-Level Anomaly Screening",
    labels={
        "mean_age_days": "Mean age (days)",
        "review_rate_pct": "Review rate (%)",
        "anomaly_flag": "Anomaly flagged"
    }
)
fig.update_traces(textposition="top center")
fig.show()

# 18. Topic metrics heatmap
heatmap = topic_summary.set_index("topic")[[
    "mean_health", "mean_freshness",
    "review_rate_pct", "mean_source_quality"
]]

fig = px.imshow(
    heatmap,
    aspect="auto",
    color_continuous_scale="RdYlGn",
    title="6. Topic Health Heatmap",
    labels={"x": "Metric", "y": "Topic", "color": "Score"}
)
fig.show()

# 19. Interactive evidence graph
positions = nx.spring_layout(G, seed=SEED, k=0.5)

edge_x, edge_y = [], []
for u, v in G.edges():
    x0, y0 = positions[u]
    x1, y1 = positions[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edges = go.Scatter(
    x=edge_x, y=edge_y,
    mode="lines",
    line=dict(width=0.7, color="#9ca3af"),
    hoverinfo="none"
)

node_x, node_y, node_text, node_color, node_size = [], [], [], [], []

for node, attrs in G.nodes(data=True):
    x, y = positions[node]
    node_x.append(x)
    node_y.append(y)
    node_text.append(
        f"{attrs.get('kind', 'Node')}: {attrs.get('label', node)}"
    )
    node_color.append(attrs.get("health", 65))
    node_size.append(13 if attrs.get("kind") == "Topic" else 8)

nodes = go.Scatter(
    x=node_x, y=node_y,
    mode="markers",
    text=node_text,
    hoverinfo="text",
    marker=dict(
        size=node_size,
        color=node_color,
        colorscale="Viridis",
        showscale=True,
        colorbar={"title": "Health / node"},
        line=dict(width=0.6, color="white")
    )
)

fig = go.Figure(data=[edges, nodes])
fig.update_layout(
    title="7. Evidence and Knowledge Relationship Graph",
    showlegend=False,
    hovermode="closest",
    height=650,
    xaxis={"visible": False},
    yaxis={"visible": False},
    margin=dict(l=10, r=10, t=45, b=10)
)
fig.show()

# 20. Candidate conflict report
display(Markdown("## Potential contradictions — human review required"))

if conflicts.empty:
    display(Markdown("No candidate pairs detected by the rule-based screen."))
else:
    display(conflicts)

# 21. Prioritized review queue
display(Markdown("## Highest-priority claims to recheck"))

review_queue = df[df["needs_review"]].sort_values(
    ["priority_score", "importance"],
    ascending=False
)

display(review_queue[[
    "claim_id", "topic", "claim", "source",
    "updated_at", "age_days", "source_quality",
    "freshness_score", "health_score", "priority_score",
    "priority_band", "potential_conflict", "review_reason"
]].round(2).head(30))

# 22. Topic summary table
display(Markdown("## Topic-level summary"))
display(topic_summary.sort_values(
    "review_rate_pct", ascending=False
).round(2))

# 23. Executive report
display(Markdown(f"""
## Executive Report

- Total claims analyzed: **{len(df)}**
- Claims needing review: **{int(df['needs_review'].sum())}**
- Claims older than 180 days: **{int((df['age_days'] > 180).sum())}**
- Low-rated sources: **{int((df['source_quality'] <= 2).sum())}**
- Candidate conflict pairs: **{len(conflicts)}**
- Mean health score: **{df['health_score'].mean():.1f}/100**
- High-priority claims: **{int((df['priority_band'] == 'High').sum())}**
- Topics flagged as anomalies: **{', '.join(topic_summary.loc[topic_summary['anomaly_flag'], 'topic'].astype(str)) or 'None'}

An anomaly is an investigation lead, not proof of a problem.
All findings above describe the synthetic demonstration dataset.
"""))

# 24. Export data
df.to_csv("knowledge_claims_analysis.csv", index=False)
topic_summary.to_csv("topic_summary.csv", index=False)
conflicts.to_csv("potential_conflicts.csv", index=False)
review_queue.to_csv("review_queue.csv", index=False)

display(Markdown("""
## Export files created
- `knowledge_claims_analysis.csv`
- `topic_summary.csv`
- `potential_conflicts.csv`
- `review_queue.csv`
"""))

# Download automatically when running in Google Colab.
try:
    from google.colab import files
    files.download("knowledge_claims_analysis.csv")
    files.download("topic_summary.csv")
    files.download("potential_conflicts.csv")
    files.download("review_queue.csv")
except Exception:
    print("CSV reports saved in the notebook's working directory.")

print("Analysis completed successfully.")
print("Use primary sources and human review before relying on any flagged claim.")



# Adaptive Knowledge Intelligence
### Evidence monitoring, freshness analysis and review prioritization

**Important:** This notebook uses synthetic demonstration data.
The scores are heuristics, not probabilities that claims are true.


## Potential contradictions — human review required

,claim_a,claim_b,topic,claim_a_text,claim_b_text,review_note
0,C003,C004,Security,The sample service accepts requests without au...,The sample service requires authentication for...,"Compare context, dates and source evidence."
1,C005,C006,Biology,A gene observed in two species proves direct t...,Shared gene occurrence alone cannot establish ...,"Compare context, dates and source evidence."
2,C010,C011,Statistics,Correlation proves that one variable causes an...,Correlation alone does not establish causation.,"Compare context, dates and source evidence."
3,C012,C013,Machine Learning,A high training score guarantees good unseen p...,Held-out evaluation estimates generalization p...,"Compare context, dates and source evidence."
4,C014,C015,API,The API returns a JSON response for valid requ...,The API returns an XML response for valid requ...,"Compare context, dates and source evidence."


## Highest-priority claims to recheck

,claim_id,topic,claim,source,updated_at,age_days,source_quality,freshness_score,health_score,priority_score,priority_band,potential_conflict,review_reason
4,C005,Biology,A gene observed in two species proves direct t...,Unreviewed note,2024-12-30,650,1,8.18,12.32,93.22,High,True,Old verification date; Low source-quality rati...
9,C010,Statistics,Correlation proves that one variable causes an...,Unreviewed note,2025-08-17,420,1,19.84,19.90,89.06,High,True,Old verification date; Low source-quality rati...
2,C003,Security,The sample service accepts requests without au...,Legacy specification,2025-05-29,500,2,14.58,23.48,77.09,High,True,Old verification date; Low source-quality rati...
11,C012,Machine Learning,A high training score guarantees good unseen p...,Old experiment note,2025-09-16,390,2,22.27,28.48,74.34,High,True,Old verification date; Low source-quality rati...
14,C015,API,The API returns an XML response for valid requ...,Old API note,2026-01-24,260,2,36.74,37.88,74.16,High,True,Old verification date; Low source-quality rati...
35,C036,Data Quality,Illustrative knowledge statement 36 about data...,Technical report,2025-01-13,636,3,8.64,26.61,65.36,High,False,Old verification date; Low health score
38,C039,Security,Illustrative knowledge statement 39 about secu...,Official documentation,2024-09-02,769,4,5.18,31.36,62.75,Medium,False,Old verification date; Low health score
64,C065,API,Illustrative knowledge statement 65 about api.,Official documentation,2025-12-02,313,2,29.96,33.47,61.59,Medium,False,Old verification date; Low source-quality rati...
16,C017,API,Illustrative knowledge statement 17 about api.,Technical report,2024-10-07,734,3,5.92,24.85,61.33,Medium,False,Old verification date; Low health score
98,C099,Statistics,Illustrative knowledge statement 99 about stat...,Dataset metadata,2026-03-09,216,1,43.53,35.29,60.59,Medium,False,Old verification date; Low source-quality rati...


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Analysis completed successfully.
Use primary sources and human review before relying on any flagged claim.
